In [1]:
sys.path.append("/mnt/c/Users/sap98/OneDrive/Escritorio/My_Stuff/programming/Uniandes/HCIM/P1/Code_lab/Self/ToyMaker/Prototype")
%load_ext line_profiler

import numpy as np
from ToyMaker import *
import sys


def calculate_propensities(propensities, species, reactions_species_index):
    """ 
    Calculates the propensity of each reaction
    Args:
        propensities (list): List of functions
        species (list): List of species
        species_index (list): List of species indexes
    Returns:
        tauarr (list): List of propensities
    """
    species_values = [species[reactions_species_index[i]] for i in range(len(reactions_species_index))]
    pt = [propensities[i](*species_values[i]) for i in range(len(propensities))]
    τarr = [calculate_tau(p) if p > 0 else np.inf for p in pt]
    return τarr


def gillespie(species, tmax, reaction_type, propensities, reactions_species_index):
    """
    
    """
    while species[0] < tmax:
        τarr = calculate_propensities(propensities, species, reactions_species_index)
        τ,  q = minimal_value(τarr)
        species = species + reaction_type[q]
        species[0] = species[0] + τ
    return species


def simulate_cell(species, reactions, tmax, sampling_time, cell=1):

    species_names = get_species_names(species)

    reaction_type = change_matrix(reactions, species_names)

    system, system_parameters, system_idx = get_funct_get_pams(species)
    propensities, propensities_parameters, params_idx = get_funct_get_pams(reactions)

    species = set_values(species)
    reactions = set_values(reactions)

    species_index = get_index(species_names, system_parameters, system)
    reactions_species_index = get_index(species_names, propensities_parameters, propensities)

    tarr = np.arange(0, tmax , sampling_time ,dtype=float) 
    sim = setup_sim(tarr, species, cell)

    for i in range(1,len(tarr)):
        sim[i] = gillespie(sim[i - 1], tarr[i], reaction_type, propensities, reactions_species_index)
        sim[i][0] = round(sim[i][0], 5)
        sim[i][1] = cell
        
    return sim

if __name__ == '__main__':

    def k_r():
        return 1

    def gamma_r():
        return 1/5

    species = {
                't':    0., 
                'cell': 0,
                'Xr': 0.,
    }
    reactions = {

        k_r: {'create': ['Xr']},
        gamma_r:{'destroy': ['Xr']},

    }


    tmax = 100
    sampling_time = 0.1
    cell = 1
    cell_sim = simulate_cell(species, reactions, tmax, sampling_time, cell=1)

In [2]:
%lprun -f simulate_cell simulate_cell(species, reactions, tmax, sampling_time, cell=1)

Timer unit: 1e-06 s

Total time: 0.033816 s
File: /tmp/ipykernel_14409/2892930367.py
Function: simulate_cell at line 37

Line #      Hits         Time  Per Hit   % Time  Line Contents
    37                                           def simulate_cell(species, reactions, tmax, sampling_time, cell=1):
    38                                           
    39         1          8.0      8.0      0.0      species_names = get_species_names(species)
    40                                           
    41         1         41.0     41.0      0.1      reaction_type = change_matrix(reactions, species_names)
    42                                           
    43         1         20.0     20.0      0.1      system, system_parameters, system_idx = get_funct_get_pams(species)
    44         1         12.0     12.0      0.0      propensities, propensities_parameters, params_idx = get_funct_get_pams(reactions)
    45                                           
    46         1          5.0      5.0

In [3]:
species_names = get_species_names(species)

reaction_type = change_matrix(reactions, species_names)

system, system_parameters, system_idx = get_funct_get_pams(species)
propensities, propensities_parameters, params_idx = get_funct_get_pams(reactions)

species = set_values(species)
reactions = set_values(reactions)

species_index = get_index(species_names, system_parameters, system)
reactions_species_index = get_index(species_names, propensities_parameters, propensities)

tarr = np.arange(0, tmax , sampling_time ,dtype=float) 
sim = setup_sim(tarr, species, cell)

%lprun -f gillespie [gillespie(sim[i - 1], tarr[i], reaction_type, propensities, reactions_species_index) for i in range(1,len(tarr))]

Timer unit: 1e-06 s

Total time: 1.21153 s
File: /tmp/ipykernel_14409/2892930367.py
Function: gillespie at line 25

Line #      Hits         Time  Per Hit   % Time  Line Contents
    25                                           def gillespie(species, tmax, reaction_type, propensities, reactions_species_index):
    26                                               """
    27                                               
    28                                               """
    29     61849      23876.0      0.4      2.0      while species[0] < tmax:
    30     60850     942937.0     15.5     77.8          τarr = calculate_propensities(propensities, species, reactions_species_index)
    31     60850     111997.0      1.8      9.2          τ,  q = minimal_value(τarr)
    32     60850      86450.0      1.4      7.1          species = species + reaction_type[q]
    33     60850      46007.0      0.8      3.8          species[0] = species[0] + τ
    34       999        260.0      0.3     

In [4]:
calculate_propensities(propensities, species, reactions_species_index)

TypeError: unhashable type: 'list'